In [14]:
import os

print("Current folder:", os.getcwd())
print("Files here:", os.listdir())
print("Parent folder:", os.listdir(".."))

Current folder: /Users/stacychan/Query Analysis/src
Files here: ['.DS_Store', '02_behavior_relationships.ipynb', '03_behavior_transitions.ipynb', '00_data_preprocessing.ipynb', '01_dataset_overview.ipynb', '04_user_behavior_profiles.ipynb']
Parent folder: ['.DS_Store', 'raw_data', 'processed_data', 'README.md', '.git', 'src']


In [15]:
import pandas as pd

df = pd.read_excel("../raw_data/Query_Analysis.xlsx")

df.to_csv("Reconciled_Data.csv", index=False)


In [17]:
import os
import pandas as pd
import numpy as np

file_path = "Reconciled_Data.csv"

# save outputs into processed_data folder
output_dir = "../processed_data"
os.makedirs(output_dir, exist_ok=True)

output_full = os.path.join(output_dir, "Reconciled_Data_Full.csv")
output_final = os.path.join(output_dir, "Reconciled_Final.csv")

# read raw csv
raw = pd.read_csv(file_path, header=None)

# actual data starts after the first 2 rows
df = raw.iloc[2:].copy().reset_index(drop=True)

# assign columns manually based on the file structure
df.columns = [
    "participant_no",
    "week_no",
    "query",

    "info_type_sophie",
    "info_type_ari",
    "info_type_bowen",
    "info_type_lexie",
    "info_type_agreement",
    "info_type_reconciliation",

    "task_sophie",
    "task_rachel",
    "task_bowen",
    "task_lexie",
    "task_agreement",
    "task_reconciliation",

    "goal_sophie",
    "goal_ari",
    "goal_bowen",
    "goal_lexie",
    "goal_agreement",
    "goal_reconciliation",
]

# turn empty strings into NA
df = df.replace(r"^\s*$", pd.NA, regex=True)

# fill merged-like blanks in participant/week
df["participant_no"] = df["participant_no"].ffill()
df["week_no"] = df["week_no"].ffill()

def is_yes(x):
    return pd.notna(x) and str(x).strip().upper() == "Y"

def fill_within_group(series):
    return series.ffill().bfill()

def first_mode(row_vals):
    vals = [str(v).strip() for v in row_vals if pd.notna(v) and str(v).strip() != ""]
    if not vals:
        return pd.NA
    return pd.Series(vals).value_counts().index[0]

group_keys = ["participant_no", "week_no", "query"]

# --------------------------------------------------
# 1) first fill reconciliation within repeated query rows
# --------------------------------------------------
for col in [
    "info_type_reconciliation",
    "task_reconciliation",
    "goal_reconciliation",
]:
    df[col] = df.groupby(group_keys, dropna=False)[col].transform(fill_within_group)

# --------------------------------------------------
# 2) if agreed = N but reconciliation is still blank,
#    fill reconciliation from coder columns
#    (majority vote among available coders)
# --------------------------------------------------

# info type fallback
mask = (~df["info_type_agreement"].apply(is_yes)) & (df["info_type_reconciliation"].isna())
df.loc[mask, "info_type_reconciliation"] = df.loc[
    mask,
    ["info_type_lexie", "info_type_bowen", "info_type_ari", "info_type_sophie"]
].apply(first_mode, axis=1)

# task fallback
mask = (~df["task_agreement"].apply(is_yes)) & (df["task_reconciliation"].isna())
df.loc[mask, "task_reconciliation"] = df.loc[
    mask,
    ["task_lexie", "task_bowen", "task_rachel", "task_sophie"]
].apply(first_mode, axis=1)

# goal fallback
mask = (~df["goal_agreement"].apply(is_yes)) & (df["goal_reconciliation"].isna())
df.loc[mask, "goal_reconciliation"] = df.loc[
    mask,
    ["goal_sophie", "goal_lexie", "goal_bowen", "goal_ari"]
].apply(first_mode, axis=1)

# --------------------------------------------------
# 3) apply final rule
#    agreed = Y:
#       type/task from Lexie
#       goal from Sophie
#    agreed = N:
#       all from Reconciliation
# --------------------------------------------------
df["final_type"] = np.where(
    df["info_type_agreement"].apply(is_yes),
    df["info_type_lexie"],
    df["info_type_reconciliation"]
)

df["final_task"] = np.where(
    df["task_agreement"].apply(is_yes),
    df["task_lexie"],
    df["task_reconciliation"]
)

df["final_goal"] = np.where(
    df["goal_agreement"].apply(is_yes),
    df["goal_sophie"],
    df["goal_reconciliation"]
)

# --------------------------------------------------
# 4) last safety fill across repeated rows of same query
# --------------------------------------------------
for col in ["final_type", "final_task", "final_goal"]:
    df[col] = df.groupby(group_keys, dropna=False)[col].transform(fill_within_group)

# --------------------------------------------------
# 5) standardize "outcome" -> "outcome expectancy"
#    across all final columns
# --------------------------------------------------
for col in ["final_type", "final_task", "final_goal"]:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.replace(r"(?i)^outcome$", "outcome expectancy", regex=True)
    )

# save full file
df.to_csv(output_full, index=False)

# save final-only file
result = df[
    ["participant_no", "week_no", "query", "final_type", "final_task", "final_goal"]
].copy()

result.to_csv(output_final, index=False)

# check
print("Current folder:", os.getcwd())
print("Saved full file to:", os.path.abspath(output_full))
print("Saved final file to:", os.path.abspath(output_final))
print()
print(result.isna().sum())
print(result.head(20))

/var/folders/_p/mv7dq2pn2_57jhgzwjp8dlrw0000gn/T/ipykernel_92116/1116568674.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return series.ffill().bfill()
/var/folders/_p/mv7dq2pn2_57jhgzwjp8dlrw0000gn/T/ipykernel_92116/1116568674.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return series.ffill().bfill()
/var/folders/_p/mv7dq2pn2_57jhgzwjp8dlrw0000gn/T/ipykernel_92116/1116568674.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=Fal

Current folder: /Users/stacychan/Query Analysis/src
Saved full file to: /Users/stacychan/Query Analysis/processed_data/Reconciled_Data_Full.csv
Saved final file to: /Users/stacychan/Query Analysis/processed_data/Reconciled_Final.csv

participant_no    0
week_no           0
query             0
final_type        0
final_task        0
final_goal        0
dtype: int64
   participant_no week_no                                     query  \
0             102  Week 1      foods to avoid while losing body fat   
1             102  Week 1      foods to avoid while losing body fat   
2             102  Week 1        foods to eat that help lose weight   
3             102  Week 1        foods to eat that help lose weight   
4             102  Week 1  vegan foods to eat that help lose weight   
5             102  Week 1                    is avocado food or bad   
6             102  Week 1         is avocado too much fat with oil'   
7             102  Week 1        is it ok to eat chips occasional

In [ ]:
output_full = "Reconciled_Data_Full.csv"
output_final = "Reconciled_Final.csv"

df.to_csv(output_full, index=False)
result.to_csv(output_final, index=False)

print("Saved:", output_full)
print("Saved:", output_final)

Saved: Reconciled_Data_Full.csv
Saved: Reconciled_Final.csv


In [2]:
import os
import pandas as pd
import numpy as np

file_path = "Reconciled_Data.csv"

# save outputs into processed_data folder
output_dir = "../processed_data"
os.makedirs(output_dir, exist_ok=True)

output_full = os.path.join(output_dir, "Reconciled_Data_Full.csv")
output_final = os.path.join(output_dir, "Reconciled_Final.csv")

# read raw csv
raw = pd.read_csv(file_path, header=None)

# actual data starts after the first 2 rows
df = raw.iloc[2:].copy().reset_index(drop=True)

# assign columns manually based on the file structure
df.columns = [
    "participant_no",
    "week_no",
    "query",

    "info_type_sophie",
    "info_type_ari",
    "info_type_bowen",
    "info_type_lexie",
    "info_type_agreement",
    "info_type_reconciliation",

    "task_sophie",
    "task_rachel",
    "task_bowen",
    "task_lexie",
    "task_agreement",
    "task_reconciliation",

    "goal_sophie",
    "goal_ari",
    "goal_bowen",
    "goal_lexie",
    "goal_agreement",
    "goal_reconciliation",
]

# turn empty strings into NA
df = df.replace(r"^\s*$", pd.NA, regex=True)

# fill merged-like blanks in participant/week
df["participant_no"] = df["participant_no"].ffill()
df["week_no"] = df["week_no"].ffill()


def is_yes(x):
    return pd.notna(x) and str(x).strip().upper() == "Y"


def fill_within_group(series):
    return series.ffill().bfill()


# --------------------------------------------------
# LABEL NORMALIZATION
# --------------------------------------------------
def normalize_label(x):

    if pd.isna(x):
        return pd.NA

    val = str(x).strip().lower()
    val = " ".join(val.split())

    label_map = {

        # procedural variants
        "how-to": "how-to (procedural)",
        "procedural": "how-to (procedural)",
        "procedual": "how-to (procedural)",
        "proc": "how-to (procedural)",
        "how-to (procedural)": "how-to (procedural)",

        # psychological variants
        "psyc": "how-to (psychological)",
        "psychological": "how-to (psychological)",
        "how-to (psyc)": "how-to (psychological)",
        "how-to (psychological)": "how-to (psychological)",

        # ideas
        "ideas": "ideas/options",
        "ideas/options": "ideas/options",

        # evaluation typos
        "evaluate": "evaluation",
        "evalute": "evaluation",
        "evaluation": "evaluation",

        # outcome
        "outcome": "outcome expectancy",
        "outcome expectancy": "outcome expectancy",

        # barrier (leave k unchanged)
        "barrier": "barrier management",
        "barrier management": "barrier management",

         # task mappings you requested
        "action": "action",

        "decision": "decision making",
        "decision making": "decision making",

        "motiv": "motivational reasoning",
        "motivational": "motivational reasoning",
        "motivational reasoning": "motivational reasoning",

        "plan": "plan",
        "plna": "plan",

        # NA normalization
        "na": "na"
    }

    return label_map.get(val, val)


def first_mode(row_vals):

    vals = [normalize_label(v) for v in row_vals if pd.notna(v) and str(v).strip() != ""]
    vals = [v for v in vals if pd.notna(v)]

    if not vals:
        return pd.NA

    return pd.Series(vals).value_counts().index[0]


group_keys = ["participant_no", "week_no", "query"]

# --------------------------------------------------
# STANDARDIZE CODER LABELS FIRST
# --------------------------------------------------

label_cols = [
    "info_type_sophie", "info_type_ari", "info_type_bowen", "info_type_lexie", "info_type_reconciliation",
    "task_sophie", "task_rachel", "task_bowen", "task_lexie", "task_reconciliation",
    "goal_sophie", "goal_ari", "goal_bowen", "goal_lexie", "goal_reconciliation",
]

for col in label_cols:
    df[col] = df[col].apply(normalize_label)


# --------------------------------------------------
# 1) fill reconciliation within repeated query rows
# --------------------------------------------------

for col in [
    "info_type_reconciliation",
    "task_reconciliation",
    "goal_reconciliation",
]:
    df[col] = df.groupby(group_keys, dropna=False)[col].transform(fill_within_group)


# --------------------------------------------------
# 2) if agreed = N but reconciliation blank
#    use majority vote
# --------------------------------------------------

# info type fallback
mask = (~df["info_type_agreement"].apply(is_yes)) & (df["info_type_reconciliation"].isna())

df.loc[mask, "info_type_reconciliation"] = df.loc[
    mask,
    ["info_type_lexie", "info_type_bowen", "info_type_ari", "info_type_sophie"]
].apply(first_mode, axis=1)


# task fallback
mask = (~df["task_agreement"].apply(is_yes)) & (df["task_reconciliation"].isna())

df.loc[mask, "task_reconciliation"] = df.loc[
    mask,
    ["task_lexie", "task_bowen", "task_rachel", "task_sophie"]
].apply(first_mode, axis=1)


# goal fallback
mask = (~df["goal_agreement"].apply(is_yes)) & (df["goal_reconciliation"].isna())

df.loc[mask, "goal_reconciliation"] = df.loc[
    mask,
    ["goal_sophie", "goal_lexie", "goal_bowen", "goal_ari"]
].apply(first_mode, axis=1)


# --------------------------------------------------
# 3) apply final rule
# --------------------------------------------------

df["final_type"] = np.where(
    df["info_type_agreement"].apply(is_yes),
    df["info_type_lexie"],
    df["info_type_reconciliation"]
)

df["final_task"] = np.where(
    df["task_agreement"].apply(is_yes),
    df["task_lexie"],
    df["task_reconciliation"]
)

df["final_goal"] = np.where(
    df["goal_agreement"].apply(is_yes),
    df["goal_sophie"],
    df["goal_reconciliation"]
)


# --------------------------------------------------
# 4) final safety fill across repeated rows
# --------------------------------------------------

for col in ["final_type", "final_task", "final_goal"]:
    df[col] = df.groupby(group_keys, dropna=False)[col].transform(fill_within_group)


# --------------------------------------------------
# 5) final normalization pass
# --------------------------------------------------

for col in ["final_type", "final_task", "final_goal"]:
    df[col] = df[col].apply(normalize_label)


# --------------------------------------------------
# SAVE OUTPUTS
# --------------------------------------------------

df.to_csv(output_full, index=False)

result = df[
    ["participant_no", "week_no", "query", "final_type", "final_task", "final_goal"]
].copy()

result.to_csv(output_final, index=False)


# --------------------------------------------------
# CHECK RESULTS
# --------------------------------------------------

print("Current folder:", os.getcwd())
print("Saved full file to:", os.path.abspath(output_full))
print("Saved final file to:", os.path.abspath(output_final))

print("\nMissing values:")
print(result.isna().sum())

print("\nPreview:")
print(result.head(20))

print("\nUnique final_type labels:")
print(sorted(result["final_type"].dropna().unique()))

Current folder: /Users/stacychan/Query Analysis/src
Saved full file to: /Users/stacychan/Query Analysis/processed_data/Reconciled_Data_Full.csv
Saved final file to: /Users/stacychan/Query Analysis/processed_data/Reconciled_Final.csv

Missing values:
participant_no    0
week_no           0
query             0
final_type        0
final_task        0
final_goal        0
dtype: int64

Preview:
   participant_no week_no                                     query  \
0             102  Week 1      foods to avoid while losing body fat   
1             102  Week 1      foods to avoid while losing body fat   
2             102  Week 1        foods to eat that help lose weight   
3             102  Week 1        foods to eat that help lose weight   
4             102  Week 1  vegan foods to eat that help lose weight   
5             102  Week 1                    is avocado food or bad   
6             102  Week 1         is avocado too much fat with oil'   
7             102  Week 1        is it 